# Virtual Datasets with Icechunk and `earthaccess`

This notebook shows how to create a virtual Icechunk store from NASA granules, read it back, and update it with new granules — without copying any data.

We use the same MUR SST collection as the kerchunk-based [`virtual_data.ipynb`](virtual_data.ipynb) notebook, but persist the virtual dataset to an [Icechunk](https://icechunk.io) store instead of a kerchunk reference file.

The workflow:

1. `virtualize(granules)` builds a virtual dataset (VDS) whose variables are backed by `ManifestArray` objects.
2. `write_virtual(vds, store)` writes the VDS to an Icechunk store (here, a local directory whose virtual chunks point back at NASA's HTTPS data).
3. `open_virtual(store)` reads the store back, automatically authorizing the virtual chunk containers with your EDL credentials.
4. To update the cube, search for new granules, `virtualize()` them, and call `write_virtual(..., append_dim="time")`.

In [1]:
import earthaccess

earthaccess.login()

Auth(authenticated=True, user='earthaccess', strategy='netrc', system=urs.earthdata.nasa.gov)

## 1. Build a virtual dataset (VDS) from granules

MUR SST is a level-4, globally gridded SST product — a great candidate for virtualizing: the granules are homogeneous, CF-compliant, and share a unified grid. See the kerchunk notebook for the caveats.

In [2]:
sst_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-01-02", "2013-01-31")
)
len(sst_granules)

20

In [3]:
vds = earthaccess.virtualize(
    sst_granules, concat_dim="time", access="indirect", load=False
)
vds

<xarray.Dataset> Size: 121GB
Dimensions:           (time: 31, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 248B 2013-01-02T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    sea_ice_fraction  (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    analysed_sst      (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
    analysis_error    (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

Notice the variables are backed by `ManifestArray` — byte-range references to the original netCDF files, not copies of the data.

## 2. Write the VDS to an Icechunk store

`write_virtual()` creates a local Icechunk repository at `mur_sst.icechunk`. The virtual chunk containers are derived from the granules' HTTPS URLs, so at read time `open_virtual()` knows these containers point at NASA data and authorizes them with your EDL token.

In [4]:
earthaccess.write_virtual(vds, "mur_sst", format="icechunk")

  2026-09-01T22:20:15.821522Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329

  2026-09-01T22:20:15.823652Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:28



<xarray.Dataset> Size: 121GB
Dimensions:           (time: 31, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 248B 2013-01-02T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    sea_ice_fraction  (time, lat, lon) int8 20GB ManifestArray<shape=(31, 179...
    analysed_sst      (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
    analysis_error    (time, lat, lon) int16 40GB ManifestArray<shape=(31, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

## 3. Read the store back

`open_virtual()` detects the Icechunk store and returns an xarray dataset that reads straight from the original NASA files (no data was copied into the store).

In [5]:
ds = earthaccess.open_virtual("mur_sst")
print(f"Dataset spanning from {ds.time.min().values} to {ds.time.max().values}")

Dataset spanning from 2013-01-02T09:00:00.000000000 to 2013-02-01T09:00:00.000000000


  2026-09-01T22:20:15.925720Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



## 4. Update the cube with new granules

Search for granules in the following time step, virtualize them, and append them to the same store along the `time` dimension.

In [6]:
new_granules = earthaccess.search_data(
    concept_id="C1996881146-POCLOUD", temporal=("2013-02-01", "2013-02-28")
)
len(new_granules)

20

In [7]:
delta_vds = earthaccess.virtualize(
    new_granules, concat_dim="time", access="indirect", load=False
)
delta_vds

<xarray.Dataset> Size: 113GB
Dimensions:           (time: 29, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 232B 2013-02-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 19GB ManifestArray<shape=(29, 179...
    sea_ice_fraction  (time, lat, lon) int8 19GB ManifestArray<shape=(29, 179...
    analysed_sst      (time, lat, lon) int16 38GB ManifestArray<shape=(29, 17...
    analysis_error    (time, lat, lon) int16 38GB ManifestArray<shape=(29, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

### Now we write the delta to our store. We can still override the icechunk kwargs if we want to.

In [8]:
earthaccess.write_virtual(delta_vds, "mur_sst", format="icechunk", append_dim="time")

  2026-09-01T22:20:40.454403Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



<xarray.Dataset> Size: 113GB
Dimensions:           (time: 29, lat: 17999, lon: 36000)
Coordinates:
  * time              (time) datetime64[ns] 232B 2013-02-01T09:00:00 ... 2013...
  * lat               (lat) float32 72kB -89.99 -89.98 -89.97 ... 89.98 89.99
  * lon               (lon) float32 144kB -180.0 -180.0 -180.0 ... 180.0 180.0
Data variables:
    mask              (time, lat, lon) int8 19GB ManifestArray<shape=(29, 179...
    sea_ice_fraction  (time, lat, lon) int8 19GB ManifestArray<shape=(29, 179...
    analysed_sst      (time, lat, lon) int16 38GB ManifestArray<shape=(29, 17...
    analysis_error    (time, lat, lon) int16 38GB ManifestArray<shape=(29, 17...
Attributes: (12/42)
    Conventions:                CF-1.5
    title:                      Daily MUR SST, Final product
    summary:                    A merged, multi-sensor L4 Foundation SST anal...
    references:                 http://podaac.jpl.nasa.gov/Multi-scale_Ultra-...
    institution:                Jet Propulsion Laboratory
    history:                    created at nominal 4-day latency; replaced nr...
    ...                         ...
    project:                    NASA Making Earth Science Data Records for Us...
    publisher_name:             GHRSST Project Office
    publisher_url:              http://www.ghrsst.org
    publisher_email:            ghrsst-po@nceo.ac.uk
    processing_level:           L4
    cdm_data_type:              grid

In [9]:
updated_ds = earthaccess.open_virtual("mur_sst")
print(
    f"Dataset now spanning from {updated_ds.time.min().values} to "
    f"{updated_ds.time.max().values}"
)

Dataset now spanning from 2013-01-02T09:00:00.000000000 to 2013-03-01T09:00:00.000000000


  2026-09-01T22:20:40.585881Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



## Summary

- `virtualize()` turns granules into a virtual dataset (no data downloaded).
- `write_virtual(vds, store)` persists it to an Icechunk store (local disk or `s3://`, with write credentials for object stores).
- `write_virtual(vds, store, append_dim="time")` appends newly virtualized granules to the cube.
- `open_virtual(store)` restores the cube, automatically authorizing the virtual chunk containers that point at NASA's HTTP/S3 data behind EDL authentication.

The store is versioned, so every update is a new commit on the `main` branch — you can inspect history, roll back, or share the store with collaborators who have access to the underlying data.